## Import Python Libraries and load environment variables

In [ ]:
import os
import urllib.request
import urllib.error
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import display, Markdown

load_dotenv()

## Create llm (model) instance and test with simple question

In [ ]:
llm = ChatOpenAI(
    base_url=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    model=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
    timeout=300,
    temperature=0.5,
    max_tokens=25000
)

In [ ]:
llm.invoke("What is the capital of South Korea?").content

## Create a (langchain) Agent instance and test with simple question

In [ ]:
agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant",
)

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's KRW worth in USD right now?"}]}
)
print(result["messages"][-1].content_blocks)

## Create a helper function to beautify llm responses in our Jupyter Notebook

In [ ]:
def bfmsgs(result):
    """Extract and display text content from agent result messages."""
    text = "\n\n".join(
        block["text"] for block in result["messages"][-1].content_blocks
        if block.get("type") == "text"
    )
    display(Markdown(text))

In [ ]:
bfmsgs(result)

## Create a dummy function that returns 1500 as KRW > USD FX

In [ ]:
def get_fx(currency: str) -> str:
    """Get exchange rate for a given currency."""
    if currency.upper() in ["KRW", "USD"]:
        return "1500"

In [ ]:
agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant",
    tools=[get_fx]
)

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's USD worth in KRW right now?"}]}
)
bfmsgs(result)

In [ ]:
# @tool  # Uncomment this line to register the function as a LangChain tool
def fetch_text_from_url(url: str) -> str:
    """Fetch the document from a URL and extract text from the <body> tag."""
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; quickstart-research/1.0)"},
    )

    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            raw = resp.read()
    except urllib.error.URLError as e:
        return f"Fetch failed: {e}"

    # Parse HTML and read content from <body>
    text = raw.decode("utf-8", errors="replace")
    soup = BeautifulSoup(text, "html.parser")
    body = soup.body

    if body:
        return body.get_text(separator="\n", strip=True)
    return "Body tag not found"

In [ ]:
fetch_text_from_url("http://hfspn.co/")

In [ ]:
language = "English"
SYSTEM_PROMPT = f"""You are a menu finder assistant. You can find menu information and prices from restaurant websites in Korea and provide it to users. Always answer in {language} and use the following tools below to fetch menu information.

## Capabilities

- `fetch_text_from_url`: loads website text from a URL into the conversation. Be aware of the structure of a website and extract only relevant information."""

In [ ]:
agent = create_agent(
    model=llm,
    tools=[fetch_text_from_url],
    system_prompt=SYSTEM_PROMPT
)

- https://www.hanyang.ac.kr/re12
- http://hfspn.co/

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's on the menu today at university campuses? Find it at https://www.hanyang.ac.kr/re12"}]}
)
bfmsgs(result)